# ✈️ PROJETO DE ENGENHARIA DE DADOS: PIPELINE ETL (ANAC ➔ POSTGRESQL)

Este notebook executa o processo completo de **ETL (Extract, Transform, Load)** dos dados públicos de ocorrências aeronáuticas da ANAC.

---

### 📋 Roteiro Passo a Passo das Etapas:
1. **Passo 1: Extração (Extract)** ➔ Ler o arquivo JSON de ocorrências e carregar na memória como DataFrame do Pandas.

2. **Passo 2: Inspeção das Colunas** ➔ Listar os nomes de todas as colunas originais do arquivo para conferência.

3. **Passo 3: Filtragem de Colunas** ➔ Selecionar apenas as 7 colunas essenciais para o nosso projeto.

4. **Passo 4: Padronização e Limpeza** ➔ Renomear colunas para remover acentos (`Classificacao_da_Ocorrência` ➔ `Classificacao_da_Ocorrencia`).

5. **Passo 5: Validação Visual** ➔ Inspecionar as primeiras linhas do DataFrame tratado antes do envio.

6. **Passo 6: Carga Completa no PostgreSQL (Load)**:

   - **6.1 a 6.2:** Configurar parâmetros e abrir conexão (`psycopg2.connect` e `cursor`).
   - **6.3:** Garantir que a tabela `Anac` existe no banco (`CREATE TABLE IF NOT EXISTS`).
   - **6.4:** **Delete da base antes da carga** (`TRUNCATE TABLE Anac`) para não acumular ou duplicar linhas.
   - **6.5:** Inserir todas as 12.221 linhas do DataFrame na tabela usando laço de repetição (`for`).
   - **6.6:** Salvar as alterações permanentemente no banco (`conn.commit()`).
   - **6.7:** Validar se a quantidade de linhas no banco bate 100% com o DataFrame.
   - **6.8:** Fechar o cursor e a conexão de forma segura (`cur.close()` e `conn.close()`).


In [ ]:
# ==============================================================================
# PASSO 1: LEITURA E EXTRAÇÃO DO ARQUIVO JSON
# ==============================================================================

# Importa a biblioteca 'pandas' e define o apelido padrão 'pd'
import pandas as pd

# Configura o pandas para exibir todas as colunas no resultado, sem ocultar com '...'
pd.set_option('display.max_columns', None)

# Armazena o caminho absoluto do arquivo JSON no seu computador
# O prefixo 'r' define uma raw string, evitando erros com caracteres de escape ou barras
caminho_do_arquivo = r"/Users/analeia/Desktop/Engenharia de Dados/V_OCORRENCIA_AMPLA.json"

# Carrega o arquivo JSON em memória como uma tabela chamada DataFrame (variável 'db')
# encoding='utf-8-sig' trata corretamente acentos e caracteres especiais da língua portuguesa
db = pd.read_json(caminho_do_arquivo, encoding='utf-8-sig')

# Exibe as 10 primeiras linhas da tabela para confirmar que os dados foram carregados
db.head(10)


In [ ]:
# ==============================================================================
# PASSO 2: INSPEÇÃO DAS COLUNAS ORIGINAIS
# ==============================================================================

# Lista os nomes de todas as colunas presentes no arquivo original da ANAC
# Esse passo é essencial para conferirmos a grafia exata de cada coluna antes de selecionar ou renomear
db.columns


In [ ]:
# ==============================================================================
# PASSO 3: SELEÇÃO DAS COLUNAS DE INTERESSE
# ==============================================================================

# Cria uma lista contendo exatamente as 7 colunas que queremos levar para o nosso banco de dados
colunas = [
    "Numero_da_Ocorrencia",         # Identificador único da ocorrência aeronáutica
    "Classificacao_da_Ocorrência",  # Categoria da gravidade (ainda com acento na origem)
    "Data_da_Ocorrencia",           # Data em que o fato aconteceu
    "Municipio",                    # Município/cidade da ocorrência
    "UF",                           # Estado / Unidade da Federação
    "Regiao",                       # Região do Brasil (ex.: Sudeste, Sul, Norte...)
    "Nome_do_Fabricante"            # Fabricante da aeronave (ex.: Embraer, Boeing...)
]

# Proteção contra reexecução: caso você execute esta célula novamente após já ter rodado
# a Célula 4, a coluna com acento não existirá mais. Esta checagem evita o erro 'KeyError':
if 'Classificacao_da_Ocorrência' not in db.columns and 'Classificacao_da_Ocorrencia' in db.columns:
    colunas[1] = 'Classificacao_da_Ocorrencia'

# Filtra o DataFrame 'db' mantendo apenas as 7 colunas da lista acima e descartando o resto
db = db[colunas]


In [ ]:
# ==============================================================================
# PASSO 4: TRATAMENTO E PADRONIZAÇÃO DE NOMES (REMOVER ACENTOS)
# ==============================================================================

# Boas práticas em Engenharia de Dados: nunca use acentos ou espaços em nomes de colunas no banco!
# Aqui renomeamos 'Classificacao_da_Ocorrência' (com circunflexo) para 'Classificacao_da_Ocorrencia' (sem acento)
if 'Classificacao_da_Ocorrência' in db.columns:
    db = db.rename(columns={'Classificacao_da_Ocorrência': 'Classificacao_da_Ocorrencia'})


In [ ]:
# ==============================================================================
# PASSO 5: VALIDAÇÃO VISUAL DOS DADOS TRATADOS
# ==============================================================================

# Exibe as 10 primeiras linhas do DataFrame já filtrado e com as colunas padronizadas
# Observe que agora a coluna se chama 'Classificacao_da_Ocorrencia' (sem acento)
db.head(10)


In [ ]:
# ==============================================================================
# PASSO 6: CARGA COMPLETA NO BANCO DE DADOS POSTGRESQL (COM LIMPEZA PRÉVIA)
# ==============================================================================

# Importa a biblioteca 'psycopg2' (driver conector entre Python e PostgreSQL)
import psycopg2

# ------------------------------------------------------------------------------
# 6.1 PARÂMETROS DE CONEXÃO COM O BANCO DE DADOS
# ------------------------------------------------------------------------------
dbname = 'python'      # Nome do banco de dados no PostgreSQL
user = 'analeia'       # Usuário cadastrado no PostgreSQL
password = '12345'     # Senha do usuário
host = 'localhost'     # Endereço do servidor ('localhost' = máquina local)
port = '5432'          # Porta padrão do serviço PostgreSQL

# ------------------------------------------------------------------------------
# 6.2 ABERTURA DA CONEXÃO E CRIAÇÃO DO CURSOR
# ------------------------------------------------------------------------------
# Estabelece o canal de comunicação física com o banco de dados PostgreSQL
conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Cria o cursor: objeto responsável por enviar comandos SQL e buscar respostas no banco
cur = conn.cursor()

# ------------------------------------------------------------------------------
# 6.3 GARANTIR QUE A TABELA EXISTE ANTES DA CARGA
# ------------------------------------------------------------------------------
# Cria a tabela 'Anac' se ela ainda não existir, definindo os tipos de dados de cada coluna
cur.execute("""
CREATE TABLE IF NOT EXISTS Anac (
    Numero_da_Ocorrencia INT,
    Classificacao_da_Ocorrencia VARCHAR(50),
    Data_da_Ocorrencia DATE,
    Municipio VARCHAR(50),
    UF VARCHAR(30),
    Regiao VARCHAR(30),
    Nome_do_Fabricante VARCHAR(100)
);
""")

# ------------------------------------------------------------------------------
# 6.4 DELETE / TRUNCATE DA BASE ANTES DA CARGA (LIMPEZA PRÉVIA)
# ------------------------------------------------------------------------------
# Esse comando limpa toda a tabela antes de iniciar a nova inserção.
# Isso impede que os dados fiquem se acumulando/duplicando a cada execução do script.
# Usamos TRUNCATE TABLE porque é instantâneo e muito mais rápido que DELETE FROM em lote.
cur.execute("TRUNCATE TABLE Anac;")

# ------------------------------------------------------------------------------
# 6.5 CARGA DOS DADOS: INSERINDO TODAS AS LINHAS DO DATAFRAME
# ------------------------------------------------------------------------------
# db.iterrows() percorre cada uma das 12.221 linhas da tabela:
# 'indice' representa o índice da linha (0, 1, 2...)
# 'coluna' representa os dados daquela linha acessíveis pelo nome da coluna
for indice, coluna in db.iterrows():
    cur.execute("""
    INSERT INTO Anac (
        Numero_da_Ocorrencia,
        Classificacao_da_Ocorrencia,
        Data_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Nome_do_Fabricante
    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        coluna['Numero_da_Ocorrencia'],        # %s 1: Inteiro
        coluna['Classificacao_da_Ocorrencia'],  # %s 2: Texto
        coluna['Data_da_Ocorrencia'],           # %s 3: Data (AAAA-MM-DD)
        coluna['Municipio'],                    # %s 4: Texto
        coluna['UF'],                           # %s 5: Texto (2 letras)
        coluna['Regiao'],                       # %s 6: Texto
        coluna['Nome_do_Fabricante']            # %s 7: Texto
    ))

# ------------------------------------------------------------------------------
# 6.6 EFETIVAÇÃO DAS ALTERAÇÕES NO BANCO (COMMIT)
# ------------------------------------------------------------------------------
# Grava permanentemente todas as 12.221 inserções no PostgreSQL
conn.commit()

# ------------------------------------------------------------------------------
# 6.7 VALIDAÇÃO AUTOMÁTICA DA QUANTIDADE DE LINHAS
# ------------------------------------------------------------------------------
# Consulta o banco de dados para checar quantas linhas estão salvas na tabela Anac
cur.execute("SELECT COUNT(*) FROM Anac;")
total_banco = cur.fetchone()[0]


# ------------------------------------------------------------------------------
# 6.8 FECHAMENTO SEGURO DOS RECURSOS
# ------------------------------------------------------------------------------
# Fecha o cursor para liberar os recursos de memória do servidor
cur.close()

# Fecha a conexão com o banco de dados PostgreSQL
conn.close()
